In [ ]:
! pip install xgboost

In [2]:
import pandas as pd

In [ ]:
data = pd.read_csv("../data/data_processed_for_tree-based_models.csv")

print(data.shape)

In [ ]:
data_processed = data.sort_values(by='time_of_day')

# 80% train, 20% test
train_size = int(len(data_processed) * 0.8)

train_data = data_processed.iloc[:train_size]
test_data  = data_processed.iloc[train_size:]

X_train = train_data.drop(columns=['normality'])
y_train = train_data['normality']

X_test  = test_data.drop(columns=['normality'])
y_test  = test_data['normality']


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


y_train_enc and y_test_enc are now numeric (0, 1, 2…), ready for XGBoost.

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# Drop non-numeric/object columns (e.g. 'value_ts') before creating DMatrix
X_train_dm = X_train.select_dtypes(include=['number', 'bool', 'category']).copy()
X_test_dm  = X_test.select_dtypes(include=['number', 'bool', 'category']).copy()

# Ensure both matrices have the same columns (add missing columns with zeros)
missing_in_test = set(X_train_dm.columns) - set(X_test_dm.columns)
for c in missing_in_test:
	X_test_dm[c] = 0
missing_in_train = set(X_test_dm.columns) - set(X_train_dm.columns)
for c in missing_in_train:
	X_train_dm[c] = 0

# Align column order
X_test_dm = X_test_dm[X_train_dm.columns]

dtrain = xgb.DMatrix(X_train_dm, label=y_train_enc)
dtest  = xgb.DMatrix(X_test_dm, label=y_test_enc)

In [ ]:
from collections import Counter

# calculate imbalance ratio
counter = Counter(y_train)
scale_pos_weight = counter[0] / counter[1]  # majority/minority ratio

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',       # AUC for binary classification
    'eta': 0.1,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42,
    'scale_pos_weight': scale_pos_weight  # handle imbalance
}


In [ ]:
dtrain = xgb.DMatrix(X_train_dm, label=y_train_enc)
dtest  = xgb.DMatrix(X_test_dm, label=y_test_enc)

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=100,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=10,
    verbose_eval=10
)


[0]	train-auc:0.94805	test-auc:0.15629
[10]	train-auc:0.99879	test-auc:0.97656
[14]	train-auc:0.99975	test-auc:0.97852


In [ ]:
y_pred_prob = bst.predict(dtest)       # probability for class 1
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))


              precision    recall  f1-score   support

           0       0.97      1.00      0.98     68974
           1       0.81      0.21      0.33      2617

    accuracy                           0.97     71591
   macro avg       0.89      0.60      0.66     71591
weighted avg       0.97      0.97      0.96     71591

ROC-AUC: 0.978520085858251
